In [1]:
# Import libraries and configure Ollama

import autogen
from autogen import AssistantAgent, LLMConfig

# Select Ollama model
MODEL = "llama3.2:latest"

# Ollama configuration
llm_config = LLMConfig({"model": MODEL, "api_type": "ollama", "client_host": "http://localhost:11434",
                        "stream": False}, temperature = 0.03)


In [2]:
# Define the task
task = """Write a concise but engaging blog post about Hope AI.
Requirements:
1. Keep the blog post within 100 words.
2. Include an attractive title.
3. Use simple and engaging language.
4. Make the content informative and interesting."""


# Create Writer Agent

writer = AssistantAgent(name = "Writer", 
                        system_message = """You are a professional content writer.
                        Your responsibilities:
                        1. Write engaging and concise blog posts.
                        2. Always include an attractive title.
                        3. Follow the given word limit.
                        4. Improve your writing based on feedback.
                        5. Provide only the final polished blog post.""",
                        llm_config = llm_config)
    

In [3]:
# Generate the initial blog post
reply = writer.generate_reply(messages = [{"content": task, 
                                            "role": "user"}])

print(reply)

{'content': '**"Revolutionizing AI: The Rise of Hope AI"**\n\nHope AI is a cutting-edge artificial intelligence platform that\'s changing the game in various industries. This innovative technology uses machine learning to analyze vast amounts of data, providing unparalleled insights and predictions. With its robust features and scalability, Hope AI is being adopted by top companies worldwide. Its potential applications range from healthcare and finance to education and customer service. As AI continues to evolve, Hope AI is at the forefront, pushing boundaries and redefining what\'s possible in the world of artificial intelligence. Experience the future of AI today with Hope AI.', 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': None}


In [4]:
# Create Critic Agent

critic = AssistantAgent(name = "Critic", is_termination_msg = lambda x: (x.get("content", "").find("TERMINATE") >= 0),
                        llm_config = llm_config, 
                        system_message = """You are a professional content critic.
                        Your responsibilities:
                        1. Review the writer's blog post.
                        2. Identify grammatical mistakes.
                        3. Check clarity and readability.
                        4. Identify areas for improvement.
                        5. Provide constructive feedback.""")
                        

In [5]:
# Start Writer-Critic conversation

res = critic.initiate_chat(recipient = writer, message = task, max_turns = 2, summary_method = "last_msg")

Critic (to Writer):

Write a concise but engaging blog post about Hope AI.
Requirements:
1. Keep the blog post within 100 words.
2. Include an attractive title.
3. Use simple and engaging language.
4. Make the content informative and interesting.

--------------------------------------------------------------------------------
Writer (to Critic):

**"Unlocking Human Potential: The Rise of Hope AI"**

Hope AI is revolutionizing the way we interact with technology. This cutting-edge artificial intelligence (AI) platform uses machine learning to analyze vast amounts of data, providing insights that can inform business decisions, improve healthcare outcomes, and enhance customer experiences. With its ability to learn from human behavior, Hope AI is poised to transform industries and unlock new possibilities for innovation and growth. By harnessing the power of AI, we can create a brighter future where technology serves humanity, not the other way around. The future is here, and it's hopefu

In [6]:
# Create SEO Reviewer
SEO_reviewer = AssistantAgent(name = "SEO_Reviewer", llm_config = llm_config, 
                              system_message = """You are an SEO reviewer.
                              Your responsibilities:
                              1. Check SEO optimization.
                              2. Suggest relevant keywords.
                              3. Improve search engine visibility.
                              Provide feedback within 3 bullet points.
                              Begin your review by stating your role.""")

In [7]:
# Create Legal Reviewer
legal_reviewer = AssistantAgent(name = "Legal_Reviewer", llm_config = llm_config,
                                system_message = """You are a legal content reviewer.
                                Your responsibilities:
                                1. Identify potentially misleading claims.
                                2. Check for potential legal concerns.
                                3. Suggest improvements where necessary.
                            
                                Provide feedback within 3 bullet points.
                                Begin your review by stating your role.""")

In [8]:
# Create Ethics Reviewer
ethic_reviewer = AssistantAgent(name = "Ethic_Reviewer", llm_config = llm_config,
                                system_message = """You are an ethics reviewer.
                                Your responsibilities:
                                1. Check whether the content is ethically appropriate.
                                2. Identify biased or misleading statements.
                                3. Ensure responsible communication.
                            
                                Provide feedback within 3 bullet points.
                                Begin your review by stating your role.""")

In [9]:
# Create Meta Reviewer
meta_reviewer = AssistantAgent(name = "Meta_Reviewer", llm_config = llm_config,
                               system_message = """You are a meta reviewer.
                               Your responsibilities:
                               1. Review feedback from all reviewers.
                               2. Combine their suggestions.
                               3. Identify the most important improvements.
                               4. Provide clear final recommendations.""")

In [10]:
# Define reflection message
def reflection_message(recipient, messages, sender, config):
    return f"""
    Review the following blog post.

    Blog post:
    {recipient.chat_messages_for_summary(sender)[-1]['content']}

    Provide your feedback based on your assigned role."""

In [11]:
# Define nested review chats
review_chats = [{"recipient": SEO_reviewer, "messages": reflection_message,
                 "summary_method": "reflection_with_llm", "summary_args": {
                     "summary_prompt": """Return the review as a JSON object only: {
                     "Reviewer": "SEO_reviewer", "Review": "Your feedback"}"""}, "max_turns": 1},
                
                {"recipient": legal_reviewer, "messages": reflection_message,
                 "summary_method": "reflection_with_llm", "summary_args": {
                     "summary_prompt": """Return the review as a JSON object only: {
                     "Reviewer": "legal_reviewer", "Review": "Your feedback"}"""}, "max_turns": 1},

                {"recipient": ethic_reviewer, "messages": reflection_message,
                 "summary_method": "reflection_with_llm", "summary_args": {
                     "summary_prompt": """Return the review as a JSON object only: {
                     "Reviewer": "ethics_reviewer", "Review": "Your feedback"}"""}, "max_turns": 1},

                {"recipient": meta_reviewer, "messages": """Aggregate feedback from all reviewers
                and provide final suggestions for improving the blog post.""", "max_turns": 1}]
                 

In [12]:
# Register nested chats
critic.register_nested_chats(review_chats, trigger=writer)

In [13]:
# Execute the complete multi-agent workflow
res = critic.initiate_chat(recipient=writer, message=task, max_turns = 2, summary_method = "last_msg")

Critic (to Writer):

Write a concise but engaging blog post about Hope AI.
Requirements:
1. Keep the blog post within 100 words.
2. Include an attractive title.
3. Use simple and engaging language.
4. Make the content informative and interesting.

--------------------------------------------------------------------------------
Writer (to Critic):

**"Unlocking Human Potential: The Rise of Hope AI"**

Hope AI is revolutionizing the way we interact with technology. This cutting-edge artificial intelligence (AI) platform uses machine learning to analyze vast amounts of data, providing insights that can inform business decisions, improve healthcare outcomes, and enhance customer experiences. With its ability to learn from human behavior, Hope AI is poised to transform industries and unlock new possibilities for innovation and growth. By harnessing the power of AI, we can create a brighter future where technology serves humanity, not the other way around. The future is here, and it's hopefu

In [14]:
# Display the final output
print("Final Blog Post:")
print(res.summary)

Final Blog Post:
**Revised Blog Post**

**"Revolutionize Your Industry with Hope AI: Unlocking Human Potential"**

Hope AI is transforming industries with its cutting-edge artificial intelligence platform. By leveraging machine learning, this innovative technology analyzes vast amounts of data to provide actionable insights that inform business decision-making and drive growth. With applications in healthcare, finance, and customer service, Hope AI is poised to revolutionize the way we work. From improving patient outcomes to streamlining operations, the impact of Hope AI is undeniable. Discover how this game-changing technology can unlock new possibilities for your industry.

**Changes Made:**

* Incorporated relevant keywords such as "Hope AI", "artificial intelligence platform", "machine learning", and "business decision-making" to improve search engine visibility.
* Updated the title to be more attention-grabbing, while maintaining its informative value.
* Expanded on the content b